# Mission 2 로컬 실행 노트북

기본은 이미 준비된 공식 데이터 폴더를 사용합니다. ZIP 준비는 DATA_MODE="zip"을 명시적으로 선택할 때만 실행합니다.

Notebook 커널에 requirements.txt의 의존성을 준비하세요. 실행 Python은 현재 커널(sys.executable)이며 DCC_PYTHON으로 재정의할 수 있습니다.


In [ ]:
import os
from pathlib import Path

current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (current, *current.parents) if (p / "src/dcc_data").is_dir() and (p / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"Repository root를 찾지 못했습니다: {current}")
print("Repository:", REPO_ROOT)


In [ ]:
import subprocess
import sys

# 커널과 CLI의 Python을 맞추되, 별도 환경이 필요하면 명시적으로 지정한다.
VENV_PYTHON = os.environ.get("DCC_PYTHON", sys.executable)
if not Path(VENV_PYTHON).is_file():
    raise FileNotFoundError(f"Python executable not found: {VENV_PYTHON}")


def run_step(args: list[str]) -> None:
    """Run a scripts/*.py CLI, streaming its output live (so a long step like
    the feature build still shows progress) while guaranteeing the real
    error text is always visible if it fails -- plain
    subprocess.run(..., check=True) only raises CalledProcessError, whose
    repr does not include the script's own printed diagnostics, which made a
    couple of real failures here hard to debug from just the traceback."""
    process = subprocess.Popen(
        [VENV_PYTHON, *args],
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"실패 (exit {process.returncode}): {' '.join(args)}")

## 0. 경로 설정 -- ZIP_DIR을 실제 경로로 바꾸세요

예전에 Colab에서 이 자리를 placeholder 그대로 돌려서 조용히 일부만 처리된 적이 있었습니다. 그래서 아래 assert가 실제로 zip 파일을 찾았는지 먼저 검증합니다 -- 여기서 에러가 나면 다음 단계로 넘어가지 마세요.

In [ ]:
DATA_MODE = "existing"  # "zip"은 원본 준비를 명시적으로 요청할 때만 선택
DATA_ROOT = Path(os.environ.get("DCC_DATA_ROOT", REPO_ROOT / "data/raw_original")).expanduser().resolve()
ZIP_DIR = None  # zip 모드에서는 ZIP 폴더와 별도의 추출 대상 폴더를 직접 지정
EXTRACT_DIR = None
CACHE_DIR = REPO_ROOT / "data/local_cache"
WORKERS = 8

if DATA_MODE == "existing":
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Official data directory not found: {DATA_ROOT}")
    OUTPUT_DIR = str(DATA_ROOT)
elif DATA_MODE == "zip":
    if ZIP_DIR is None or EXTRACT_DIR is None:
        raise ValueError("zip 모드에서는 ZIP_DIR와 EXTRACT_DIR를 지정하세요")
    zip_dir = Path(ZIP_DIR).expanduser().resolve()
    extract_dir = Path(EXTRACT_DIR).expanduser().resolve()
    if extract_dir == DATA_ROOT or DATA_ROOT in extract_dir.parents or extract_dir in DATA_ROOT.parents:
        raise ValueError("추출 대상은 기존 원본 데이터와 별도 위치여야 합니다")
    if extract_dir == zip_dir or extract_dir in zip_dir.parents or zip_dir in extract_dir.parents:
        raise ValueError("ZIP 폴더와 추출 대상은 별도 위치여야 합니다")
    if not zip_dir.is_dir() or not list(zip_dir.rglob("*.zip")):
        raise FileNotFoundError(f"ZIP files not found: {zip_dir}")
    ZIP_DIR = str(zip_dir)
    OUTPUT_DIR = str(extract_dir)
else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE}")
print("Data:", OUTPUT_DIR)
print("Cache:", CACHE_DIR)


In [ ]:
if DATA_MODE == "zip":
    import shutil
    # C드라이브가 아니라 실제 추출 대상 파일시스템의 여유 공간을 확인한다.
    existing_parent = next(p for p in (Path(OUTPUT_DIR), *Path(OUTPUT_DIR).parents) if p.exists())
    free_gb = shutil.disk_usage(existing_parent).free / 1e9
    source_gb = sum(p.stat().st_size for p in zip_dir.rglob("*") if p.is_file()) / 1e9
    print(f"추출 대상 여유: {free_gb:.1f} GB; 원본 용량: {source_gb:.1f} GB")
    if free_gb < source_gb * 1.1:
        print("[경고] 압축 해제 후 실제 용량은 더 클 수 있습니다. 여유 공간을 확인하세요.")


## 1. 선택적 ZIP 추출 (Training 전체 + Validation 50)

existing 모드에서는 추출하지 않습니다. ZIP 모드는 기존 표본 조건을 유지합니다.


In [ ]:
if DATA_MODE == "zip":
    run_step(
        [
            "scripts/extract_dcc_sample_from_zips.py",
            "--zip-dir", ZIP_DIR,
            "--output-dir", OUTPUT_DIR,
            "--n-train-calls", "0",
            "--n-val-calls", "50",
            "--seed", "42",
        ]
    )

ZIP 모드에서만 기존 29,250통화 준비 조건을 확인합니다. 기존 데이터 모드는 원본을 복사하거나 합치지 않습니다.


## 1b. 이미 통째로 풀려있던 chunk 병합

`대학부 데이터` 폴더 안에 zip이 아니라 이미 통째로 풀려있는 chunk가 하나 있었습니다(예전에 C드라이브에서 직접 풀다가 남은 것으로 추정). zip 스캔으로는 안 잡히니 따로 복사해서 합칩니다.

In [ ]:
if DATA_MODE == "zip":
    import shutil


    def _find_data_root(chunk: Path) -> Path | None:
        """The already-extracted chunk has Training/Validation nested one level
        under its own "대학부 데이터" subfolder -- find whichever level actually
        holds them, generically (not hardcoded to a specific chunk name)."""
        for name in ("Training", "Validation"):
            for hit in chunk.rglob(name):
                return hit.parent
        return None


    already_extracted_chunks = [
        chunk
        for chunk in sorted(p for p in zip_dir.iterdir() if p.is_dir())
        if not list(chunk.rglob("*.zip"))  # has no zip inside -> not handled by step 1
    ]

    print(f"zip 없이 이미 풀려있는 chunk: {len(already_extracted_chunks)}개")
    for chunk in already_extracted_chunks:
        data_root = _find_data_root(chunk)
        if data_root is None:
            print(f"  [주의] {chunk} 안에서 Training/Validation을 못 찾음 -- 수동 확인 필요")
            continue
        shutil.copytree(data_root, OUTPUT_DIR, dirs_exist_ok=True)
        print(f"  병합 완료: {data_root} -> {OUTPUT_DIR}")

## 2. manifest 생성

In [ ]:
manifest_path = REPO_ROOT / "data/manifests/dcc_manifest_local_full.csv"
if manifest_path.exists():
    # 기존 manifest를 새 원본과 혼용하거나 덮어쓰지 않도록 경로를 검증한다.
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))
    from dcc_data.manifest import load_manifest
    load_manifest(manifest_path, data_root=Path(OUTPUT_DIR), validate_paths=True)
    print("Existing manifest preserved:", manifest_path)
else:
    run_step(["scripts/build_manifest.py", "--data-root", OUTPUT_DIR, "--output", str(manifest_path)])


## 3. 발화 인덱스 + feature 빌드 (끊기거나 실패해도 `--resume` 붙여서 그대로 재실행하면 이어서 진행)

In [ ]:
index_output = str(CACHE_DIR / "m2_utterances_train_local_full.parquet")
features_output = str(CACHE_DIR / "m2_features_train_local_full.parquet")
failures_log = str(CACHE_DIR / "m2_build_failures_local_full.jsonl")

run_step(
    [
        "scripts/build_mission2_dataset.py",
        "--manifest", str(manifest_path),
        "--data-root", OUTPUT_DIR,
        "--n-calls", "0",
        "--seed", "42",
        "--index-output", index_output,
        "--features-output", features_output,
        "--failures-log", failures_log,
        "--workers", str(WORKERS),
        "--flush-every", "2000",  # 병렬이라 빨리 끝나므로 200마다 전체를 다시 쓰는 오버헤드를 줄임
        "--resume",
    ]
)

## 4. 학습 (기존 baseline과 동일한 GroupShuffleSplit(seed=42))

In [ ]:
model_output = str(CACHE_DIR / "m2_baseline_lgbm_local_full.pkl")
log_output = str(CACHE_DIR / "m2_exp_log_local_full.json")

run_step(
    [
        "scripts/train_mission2_baseline.py",
        "--features", features_output,
        "--model-output", model_output,
        "--log-output", log_output,
    ]
)

In [ ]:
import json
print(json.dumps(json.loads((CACHE_DIR / "m2_exp_log_local_full.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))